In [ ]:
!pip install -q qdrant-client fastembed langchain-text-splitters python-dotenv
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# Thêm vào BƯỚC 1
!pip install -q sentence-transformers transformers

In [ ]:

import json
import os
from typing import List, Optional
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import torch
from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import (
    SparseVectorParams, 
    SparseIndexParams, 
    PointStruct, 
    VectorParams, 
    Distance
)
from tqdm.auto import tqdm
import time

# ============================================================================
# BƯỚC 3: CẤU HÌNH
# ============================================================================
class Config:
    # Qdrant Cloud settings
    QDRANT_URL = ""
    QDRANT_API_KEY = ""
    COLLECTION_NAME = "legal_documents"
    
    # Embedding models - SỬA PHẦN NÀY
    # Dùng model từ HuggingFace
    EMBEDDING_MODEL = "AITeamVN/Vietnamese_Embedding_v2"  # Ví dụ model tiếng Việt
    # Hoặc: "VoVanPhuc/sup-SimCSE-VietNamese-phobert-base"
    # Hoặc: "bkai-foundation-models/vietnamese-bi-encoder"
    
    USE_FASTEMBED = False  # Đặt False để dùng HuggingFace
    SPARSE_MODEL = "Qdrant/bm25"  # Sparse vẫn dùng Qdrant
    
    # Processing settings
    CHUNK_SIZE = 2048
    CHUNK_OVERLAP = 128
    BATCH_SIZE = 64  # Batch size cho embedding
    UPSERT_BATCH_SIZE = 100  # Batch size cho upload
    
    # File path
    DATA_FILE = "/kaggle/input/datasets/layzeeqiu/qdrant-dataset-v2/documents.jsonl"  # Path đến file data

config = Config()




def load_documents_jsonl(path: str) -> list[Document]:
    documents = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)

            documents.append(
                Document(
                    page_content=record["page_content"],
                    metadata=record.get("metadata", {})
                )
            )

    return documents

class QdrantVDB:
    def __init__(self):
        self.collection_name = config.COLLECTION_NAME
        self.batch_size = config.BATCH_SIZE
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        print(f"Initializing models on {self.device.upper()}...")
        

        if config.USE_FASTEMBED:
            # Dùng FastEmbed (model có sẵn trong Qdrant)
            self.embedding_model = TextEmbedding(
                model_name=config.EMBEDDING_MODEL,
                providers=["CUDAExecutionProvider"] if self.device == "cuda" else ["CPUExecutionProvider"]
            )
            self.embed_func = self._fastembed_embed
        else:
            # Dùng SentenceTransformers (HuggingFace)
            from sentence_transformers import SentenceTransformer
            self.embedding_model = SentenceTransformer(
                config.EMBEDDING_MODEL,
                device=self.device
            )
            self.embedding_model.max_seq_length = 2048
            self.embed_func = self._huggingface_embed
        
        # Sparse embedding vẫn dùng Qdrant
        self.sparse_embedding = SparseTextEmbedding(
            model_name=config.SPARSE_MODEL
        )
        
        # Get embedding size
        test_embed = self.embed_func(["test"])[0]
        self.size_embedding = len(test_embed)
        print(f"Embedding dimension: {self.size_embedding}")
        
        self.client = None
    

    def _fastembed_embed(self, texts: List[str]) -> List:
        """Embed using FastEmbed"""
        return list(self.embedding_model.embed(texts))
    

    def _huggingface_embed(self, texts: List[str]) -> List:
        """Embed using HuggingFace SentenceTransformers"""
        embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=False,
            batch_size=self.batch_size
        )
        return embeddings
    def initialize_client(self) -> QdrantClient:
        """Khởi tạo Qdrant client"""
        try:
            print("\nConnecting to Qdrant Cloud...")
            self.client = QdrantClient(
                url=config.QDRANT_URL,
                api_key=config.QDRANT_API_KEY,
            )
            print("Connected successfully")
            return self.client
        except Exception as e:
            raise RuntimeError(f"Failed to connect to Qdrant: {e}") from e
    
    def delete_collection_if_exists(self):
        """Xóa collection cũ nếu tồn tại"""
        if self.client is None:
            raise RuntimeError("Qdrant client not initialized")
        
        try:
            if self.client.collection_exists(self.collection_name):
                print(f"Deleting existing collection '{self.collection_name}'...")
                self.client.delete_collection(self.collection_name)
                print("Collection deleted")
                time.sleep(2)  # Đợi một chút để đảm bảo xóa hoàn tất
            else:
                print(f"ℹCollection '{self.collection_name}' does not exist")
        except Exception as e:
            print(f"Warning: Could not check/delete collection: {e}")
    
    def create_collection(self):
        """Tạo collection mới"""
        if self.client is None:
            raise RuntimeError("Qdrant client not initialized")
        
        print(f"\nCreating collection '{self.collection_name}'...")
        
        try:
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config={
                    "dense": VectorParams(
                        size=self.size_embedding,
                        distance=Distance.COSINE
                    )
                },
                sparse_vectors_config={
                    "sparse": SparseVectorParams(
                        index=SparseIndexParams(on_disk=False)
                    )
                }
            )
            print(f"Collection created successfully")
        except Exception as e:
            raise RuntimeError(f"Failed to create collection: {e}") from e
    
    def embed_and_store(self, documents: List[Document]):
        """Embed documents và upload lên Qdrant"""
        if self.client is None:
            raise RuntimeError("Qdrant client not initialized")
        
        print(f"\nStarting embedding and upload process...")
        print(f"   Total documents: {len(documents)}")
        print(f"   Batch size: {self.batch_size}")
        print(f"   Upload batch size: {config.UPSERT_BATCH_SIZE}")
        
        points = []
        total_uploaded = 0
        
        with tqdm(total=len(documents), desc="Processing documents") as pbar:
            for i in range(0, len(documents), self.batch_size):
                batch = documents[i:i+self.batch_size]
                texts = [d.page_content for d in batch]
                
                # SỬA PHẦN NÀY - Generate embeddings
                dense = self.embed_func(texts)  # Dùng hàm embed động
                sparse = list(self.sparse_embedding.embed(texts))
                
                # Create points
                for j, doc in enumerate(batch):
                    # Đảm bảo dense embedding là list
                    if hasattr(dense[j], 'tolist'):
                        dense_vector = dense[j].tolist()
                    else:
                        dense_vector = dense[j]
                    
                    points.append(
                        PointStruct(
                            id=i+j,
                            vector={
                                "dense": dense_vector,
                                "sparse": {
                                    "indices": sparse[j].indices.tolist(),
                                    "values": sparse[j].values.tolist()
                                }
                            },
                            payload = {
                                "text": doc.page_content,
                                "metadata": doc.metadata
                            }

                        )
                    )
                
                # Upload when batch is full
                if len(points) >= config.UPSERT_BATCH_SIZE:
                    self.client.upsert(
                        collection_name=self.collection_name,
                        points=points,
                        wait=False
                    )
                    total_uploaded += len(points)
                    pbar.set_postfix({"uploaded": total_uploaded})
                    points = []
                
                pbar.update(len(batch))
        
        # Upload remaining points
        if points:
            self.client.upsert(
                collection_name=self.collection_name,
                points=points,
                wait=True
            )
            total_uploaded += len(points)
        
        print(f"Upload completed! Total vectors: {total_uploaded}")
    
    def run_pipeline(self):
        """Chạy toàn bộ pipeline"""
        try:

            documents = load_documents_jsonl("/kaggle/input/datasets/layzeeqiu/qdrant-dataset-v2/documents.jsonl")
            
            # 2. Connect to Qdrant
            self.initialize_client()
            
            # 3. Xóa collection cũ
            self.delete_collection_if_exists()
            
            # 4. Tạo collection mới
            self.create_collection()
            
            # 5. Embed và upload
            self.embed_and_store(documents)
            

            
            print("\nPipeline completed successfully!")
            
        except Exception as e:
            print(f"\n Error occurred: {e}")
            raise




In [ ]:

def main():
    print("=" * 70)
    print(" QDRANT EMBEDDING PIPELINE")
    print("=" * 70)
    
    # Kiểm tra GPU
    if torch.cuda.is_available():
        print(f" GPU Available: {torch.cuda.get_device_name(0)}")
        print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print(" No GPU available, using CPU")
    
    print("\n Configuration:")
    print(f"   Qdrant URL: {config.QDRANT_URL}")
    print(f"   Collection: {config.COLLECTION_NAME}")
    print(f"   Embedding Model: {config.EMBEDDING_MODEL}")
    print(f"   Data File: {config.DATA_FILE}")
    print("=" * 70)
    
    # Confirm before proceeding
    response = input("\n This will DELETE existing collection. Continue? (yes/no): ")
    if response.lower() != 'yes':
        print(" Operation cancelled")
        return
    
    # Run pipeline
    vdb = QdrantVDB()
    vdb.run_pipeline()


if __name__ == "__main__":
    main()